# 04 — Evaluate breakpoints, pick N per well

Adapted from `legacy/notebooks/03_evaluation.ipynb`.

This notebook is the human-in-the-loop step. For each well:
  1. Load the BIC-sweep JSON saved by notebook 03.
  2. Use the interactive slider to inspect 0–10 breakpoints visually.
  3. Once you've decided, set the chosen N in the dictionary at the bottom.
  4. Save the chosen breakpoints to `results/breakpoints/2022_02/<well_id>__chosen.csv`.

The interactive slider gives you R², adjusted R², and a visual on each
candidate; the elbow/min-BIC suggestion from notebook 03 is just a
starting point, not a verdict.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, json
if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd

from karst_analysis.runs import register_run
from karst_analysis.sec.breakpoints import (
    rebuild_model, extract_segments, extract_breakpoints,
    calculate_metrics_per_segment, get_breakpoint_data,
)
from karst_analysis.sec.viz import (
    interactive_segmented_regression, plot_segments,
)

In [ ]:
# Inputs
CHOICES_CSV = Path("results/chosen_method.csv")
BP_DIR      = Path("data/breakpoints/2022_02")
OUT_DIR     = Path("results/breakpoints/2022_02")
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_LOG10   = True

choices = pd.read_csv(CHOICES_CSV)
choices

In [ ]:
# Helper: load the JSON for one well, return (x, y, full results dict, df_for_slider)
def load_for_well(row):
    well_id = row["well_id"]
    date    = str(row["date"])
    csv_path = Path(row["chosen_file"])

    df_smooth = pd.read_csv(csv_path)
    x_col = "depth_m"
    y_col = "log10_sec_uS_cm" if USE_LOG10 and "log10_sec_uS_cm" in df_smooth.columns else "sec_uS_cm"
    mask = df_smooth[y_col].notna()
    x = df_smooth.loc[mask, x_col].to_numpy()
    y = df_smooth.loc[mask, y_col].to_numpy()

    # Find the most recent BIC json for this well
    candidates = sorted(BP_DIR.glob(f"{well_id}_{date}__bp-*.json"))
    if not candidates:
        raise FileNotFoundError(f"No JSON for {well_id} {date} in {BP_DIR}")
    json_path = candidates[-1]
    with open(json_path) as f:
        bic_results = json.load(f)

    # Pick trial_1 by default — slider works on its 'estimates' column
    df_for_slider = pd.DataFrame(bic_results["trial_1"]["df"])
    return x, y, bic_results, df_for_slider, json_path

## Per-well exploration

Edit the `well_idx` value below to flip between wells. For each one,
the interactive slider lets you scan 0–10 breakpoints to pick the right N.


In [ ]:
# >>> Pick a well from the choices table <<<
well_idx = 0  # change to 1, 2, ..., len(choices)-1

row = choices.iloc[well_idx]
print(f"Well: {row['well_id']}  Date: {row['date']}  Smoothing: {row['method']}")

x, y, bic_results, df_slider, json_path = load_for_well(row)
print(f"BIC JSON: {json_path.name}")
print(f"trial_1 suggests: elbow_bic={bic_results['trial_1']['best_n_breakpoint_bic']}, "
      f"min_bic_n={bic_results['trial_1']['min_bic_n_breakpoint']}")

In [ ]:
# Interactive slider — drag to compare 0..10 breakpoints
interactive_segmented_regression(
    x, y, df_slider,
    title=f"{row['well_id']} {row['date']}",
    breakpoints=int(bic_results['trial_1']['best_n_breakpoint_bic']),
)

## Once you've decided N, fit and save

Set `N_CHOSEN` to the value you picked from the slider, then run the next cell.


In [ ]:
# Set this for the current well
N_CHOSEN = 3   # <— change after evaluating slider

# Rebuild the model with the chosen N from the trial_1 estimates
trial1 = bic_results["trial_1"]["df"]
trial1_df = pd.DataFrame(trial1)

# Pull the estimates dict for the chosen N
row_for_n = trial1_df[trial1_df["n_breakpoints"] == N_CHOSEN].iloc[0]
estimates_n = row_for_n["estimates"]

params_dict = {"n_breakpoints": N_CHOSEN, "estimates": estimates_n}
fit = rebuild_model(x, y, params_dict)

bps_df = extract_breakpoints(fit)
seg_info = extract_segments(fit)
metrics = calculate_metrics_per_segment(fit)

print("Breakpoints:")
display(bps_df)
print("Segment metrics:")
display(pd.DataFrame(metrics))

plot_segments(seg_info, metrics, title=f"{row['well_id']} {row['date']} — N={N_CHOSEN}")

In [ ]:
# Save chosen breakpoints (uses register_run → ledger entry)
params = {
    "method": "breakpoints_chosen",
    "n_chosen": int(N_CHOSEN),
    "smoothing_method": row["method"],
    "y_space": "log10" if USE_LOG10 else "linear",
}

with register_run(
    stage="breakpoints_chosen",
    well_id=row["well_id"], date=str(row["date"]),
    input_file=str(json_path), params=params,
    output_dir=OUT_DIR,
) as run:
    bps_df.to_csv(run.output_path, index=False)
    run.note = f"chose N={N_CHOSEN} after visual inspection"
    print(f"Saved: {run.output_path}")

print("\nNow change well_idx to the next well and repeat.")